# Phase 2b — Feature Engineering with PySpark Window Functions

## What features are we building and why?

A machine learning model can only learn from **numbers**. We need to convert raw match history ("Brazil beat Argentina 2-1 on 15 Nov 2023") into numerical features that capture each team's **form, strength, and tendencies**.

### Feature Set

| Feature | Description | Why It Predicts Outcomes |
|---------|-------------|---------------------------|
| `win_rate_last10` | Fraction of last 10 matches won | Teams on a winning streak tend to keep winning |
| `draw_rate_last10` | Fraction of last 10 matches drawn | High draw rate → team is defensively solid but struggles to win |
| `avg_goals_scored_last20` | Avg goals scored in last 20 matches | Offensive strength signal |
| `avg_goals_conceded_last20` | Avg goals conceded in last 20 matches | Defensive fragility signal |
| `avg_goal_diff_last10` | Avg (scored − conceded) last 10 | Net dominance indicator |
| `goal_diff_trend` | Avg GD last 5 minus avg GD prev 5 | Is the team improving or declining? |
| `form_last6m` | Win rate in last 6 months | Captures current tournament preparation form |
| `form_prev6m` | Win rate in months 6–12 ago | Baseline to compare against recent form |
| `ranking_proxy` | = form_last6m (strength score) | Proxy for FIFA ranking without external data |
| `ranking_change` | form_last6m − form_prev6m | Equivalent of ranking movement |
| `h2h_win_rate` | Historical W rate vs this specific opponent | Some teams consistently beat others |
| `h2h_total_matches` | # prior meetings | Low count → h2h feature is less reliable |

## Key Concept: Window Functions

Spark **window functions** compute a value for each row based on a **sliding window of related rows**. Think of it like:
> "For THIS match, look back at THIS team's last 10 matches and compute an average"

The critical detail: we use `.rowsBetween(-10, -1)` — the `-1` means we **exclude the current row**. This prevents **data leakage** — we must never use a match's own result to predict itself.

## No FIFA Ranking CSV — Here's Why

The dataset doesn't include official FIFA rankings. We derive a proxy from match results (form-based strength score). If you add a `rankings.csv` later, you can replace `ranking_proxy` with the real ranking column.

In [ ]:
# ============================================================
# Cell 2 — Setup: SparkSession + load parquet files
# ============================================================
import os
import sys
from pathlib import Path
from dotenv import load_dotenv

# Load .env for DB credentials
load_dotenv(dotenv_path=Path("..") / ".env")

# ---- Windows: set JAVA_HOME before importing pyspark ----
_java_candidates = [
    r"C:\Program Files\Eclipse Adoptium\jdk-11.0.23.9-hotspot",
    r"C:\Program Files\Eclipse Adoptium\jdk-11.0.22.9-hotspot",
    r"C:\Program Files\Microsoft\jdk-11.0.22.7-hotspot",
    r"C:\Program Files\Java\jdk-11",
    r"C:\Program Files\Java\jdk1.8.0_401",
]
if "JAVA_HOME" not in os.environ:
    for c in _java_candidates:
        if os.path.exists(c):
            os.environ["JAVA_HOME"] = c
            break
if sys.platform == "win32" and "HADOOP_HOME" not in os.environ:
    os.environ["HADOOP_HOME"] = r"C:\hadoop"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("FIFA_WC2026_Features")
    .master("local[4]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "2g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

# ---- Load cleaned parquet ----
PROCESSED_DATA = str(Path("..") / "data" / "processed")

df_matches = spark.read.parquet(f"{PROCESSED_DATA}/matches_clean.parquet")

print(f"Loaded matches: {df_matches.count():,} rows")
print(f"Spark UI: {spark.sparkContext.uiWebUrl}")

df_matches.printSchema()

## Step 1 — Create Team-Centric View

The raw `matches_clean` table has one row per match with **home_team** and **away_team** as separate columns. 

For window functions, we need each team to appear **once per match** as the subject. We do this by:
1. Creating a "home perspective" view of every match
2. Creating an "away perspective" view of every match
3. Stacking (`UNION`) them together

Result: if there are 49,000 matches, we get ~98,000 rows — one per team per match.

In [ ]:
# ============================================================
# Cell 3 — Compute per-team window function features
# ============================================================

# ---- Step 1: Create team-centric view via UNION ----
# Home team perspective: team is home_team, opponent is away_team
home_view = df_matches.select(
    F.col("date"),
    F.col("home_team").alias("team"),
    F.col("away_team").alias("opponent"),
    F.col("home_score").alias("goals_for"),
    F.col("away_score").alias("goals_against"),
    F.when(F.col("result") == "W", 1).otherwise(0).alias("win"),
    F.when(F.col("result") == "D", 1).otherwise(0).alias("draw"),
    F.when(F.col("result") == "L", 1).otherwise(0).alias("loss"),
    F.lit("home").alias("venue"),
    F.col("neutral"),
    F.col("tournament"),
)

# Away team perspective: wins/losses are flipped
away_view = df_matches.select(
    F.col("date"),
    F.col("away_team").alias("team"),
    F.col("home_team").alias("opponent"),
    F.col("away_score").alias("goals_for"),
    F.col("home_score").alias("goals_against"),
    F.when(F.col("result") == "L", 1).otherwise(0).alias("win"),   # Away win = home loss
    F.when(F.col("result") == "D", 1).otherwise(0).alias("draw"),
    F.when(F.col("result") == "W", 1).otherwise(0).alias("loss"),  # Away loss = home win
    F.lit("away").alias("venue"),
    F.col("neutral"),
    F.col("tournament"),
)

# Stack: 2 rows per match
team_matches = home_view.union(away_view).orderBy("date")

print(f"Team-centric rows: {team_matches.count():,} (should be ~2x match count)")

# ---- Step 2: Define window specifications ----
# We order by unix timestamp (integer) so Spark can do range-based windows.
# rowsBetween(-N, -1) = look back N rows but EXCLUDE current row (no leakage)
team_matches = team_matches.withColumn(
    "ts", F.unix_timestamp(F.col("date")).cast(T.LongType())  # Seconds since epoch
)

# Row-based windows (last N matches regardless of time gaps)
w_last10 = Window.partitionBy("team").orderBy("ts").rowsBetween(-10, -1)
w_prev10 = Window.partitionBy("team").orderBy("ts").rowsBetween(-20, -11)  # 11-20 matches ago
w_last5  = Window.partitionBy("team").orderBy("ts").rowsBetween(-5,  -1)
w_prev5  = Window.partitionBy("team").orderBy("ts").rowsBetween(-10, -6)
w_last20 = Window.partitionBy("team").orderBy("ts").rowsBetween(-20, -1)

# Time-based windows (last 6 months / last 12 months by calendar time)
SIX_MONTHS  = 180 * 24 * 3600  # seconds in ~6 months
TWELVE_MONTHS = 365 * 24 * 3600

w_last6m  = Window.partitionBy("team").orderBy("ts").rangeBetween(-SIX_MONTHS, -1)
w_prev6m  = Window.partitionBy("team").orderBy("ts").rangeBetween(-TWELVE_MONTHS, -SIX_MONTHS - 1)

# ---- Step 3: Compute all rolling features with window functions ----
team_features = (
    team_matches
    # --- Recent form (last 10 matches) ---
    # avg() over a 0/1 win column = win rate (fraction of matches won)
    .withColumn("win_rate_last10",  F.avg("win").over(w_last10))
    .withColumn("draw_rate_last10", F.avg("draw").over(w_last10))
    .withColumn("loss_rate_last10", F.avg("loss").over(w_last10))

    # --- Offensive / defensive strength (last 20 matches) ---
    .withColumn("avg_goals_scored_last20",    F.avg("goals_for").over(w_last20))
    .withColumn("avg_goals_conceded_last20",  F.avg("goals_against").over(w_last20))

    # --- Goal difference per period (for trend calculation) ---
    .withColumn("gd", F.col("goals_for") - F.col("goals_against"))  # Per-match goal diff
    .withColumn("avg_gd_last5",  F.avg("gd").over(w_last5))
    .withColumn("avg_gd_prev5",  F.avg("gd").over(w_prev5))
    # Trend: positive = improving (GD getting better), negative = declining
    .withColumn(
        "goal_diff_trend",
        F.col("avg_gd_last5") - F.col("avg_gd_prev5")
    )
    .withColumn("avg_goal_diff_last10", F.avg("gd").over(w_last10))

    # --- Time-based form (proxy for ranking) ---
    .withColumn("form_last6m",  F.avg("win").over(w_last6m))   # Win rate in last 6 months
    .withColumn("form_prev6m",  F.avg("win").over(w_prev6m))   # Win rate 6-12 months ago
    # ranking_proxy = current form score (replaces external FIFA ranking data)
    .withColumn("ranking_proxy",  F.col("form_last6m"))
    # ranking_change: positive = team improving, negative = team declining
    .withColumn(
        "ranking_change",
        F.when(
            F.col("form_prev6m").isNotNull(),
            F.col("form_last6m") - F.col("form_prev6m")
        ).otherwise(F.lit(0.0))  # Default to 0 if not enough history
    )
    # Match count available (tells model how reliable the features are)
    .withColumn("matches_available", F.count("win").over(w_last20))
)

# Fill nulls with 0.5 (neutral prior) for teams with too few matches
# Using 0.5 = "unknown form" is better than 0 (which implies always losing)
form_cols = [
    "win_rate_last10", "draw_rate_last10", "loss_rate_last10",
    "avg_goals_scored_last20", "avg_goals_conceded_last20",
    "avg_gd_last5", "avg_gd_prev5", "goal_diff_trend",
    "avg_goal_diff_last10", "form_last6m", "form_prev6m",
    "ranking_proxy", "ranking_change",
]

for col_name in form_cols:
    team_features = team_features.withColumn(
        col_name,
        F.coalesce(F.col(col_name), F.lit(0.0))
    )

# Keep only the feature columns we'll join to matches
feature_cols = [
    "date", "team", "opponent", "venue",
    "win_rate_last10", "draw_rate_last10", "loss_rate_last10",
    "avg_goals_scored_last20", "avg_goals_conceded_last20",
    "avg_goal_diff_last10", "goal_diff_trend",
    "form_last6m", "form_prev6m", "ranking_proxy", "ranking_change",
    "matches_available",
]
team_features_final = team_features.select(feature_cols)

print(f"Team features rows: {team_features_final.count():,}")
print("\nSample (France):")
(
    team_features_final
    .filter(F.col("team") == "France")
    .orderBy(F.col("date").desc())
    .show(3, truncate=False)
)

## Head-to-Head (H2H) Features

**Head-to-head history** captures something form metrics can't: some teams consistently beat specific opponents regardless of current form (e.g. Germany vs. Argentina in World Cup finals historically).

For H2H we need to normalize the team pair so `(Brazil, Argentina)` and `(Argentina, Brazil)` refer to the same pairing. We do this by always putting the alphabetically-first team in the `team_a` slot.

We then store: how many times each won, and total meetings.

In [ ]:
# ============================================================
# Cell 4 — Join all features into master match_features table
# ============================================================

# ---- Part A: Compute Head-to-Head records ----

# Normalize pair: always put alphabetically-first team as 'team_a'
# This ensures (Brazil, Germany) == (Germany, Brazil)
h2h_base = df_matches.select(
    "date", "home_team", "away_team", "result"
).withColumn(
    "team_a",
    F.when(F.col("home_team") < F.col("away_team"), F.col("home_team")).otherwise(F.col("away_team"))
).withColumn(
    "team_b",
    F.when(F.col("home_team") < F.col("away_team"), F.col("away_team")).otherwise(F.col("home_team"))
).withColumn(
    # team_a_won: 1 if the alphabetically-first team won this match
    "team_a_won",
    F.when(
        (F.col("home_team") == F.col("team_a")) & (F.col("result") == "W"), 1
    ).when(
        (F.col("away_team") == F.col("team_a")) & (F.col("result") == "L"), 1
    ).otherwise(0)
).withColumn(
    "team_b_won",
    F.when(
        (F.col("home_team") == F.col("team_b")) & (F.col("result") == "W"), 1
    ).when(
        (F.col("away_team") == F.col("team_b")) & (F.col("result") == "L"), 1
    ).otherwise(0)
).withColumn(
    "is_draw", F.when(F.col("result") == "D", 1).otherwise(0)
)

# Compute CUMULATIVE H2H stats up to (but not including) each match date
# This avoids data leakage — we only use past meetings to predict this match
w_h2h = (
    Window
    .partitionBy("team_a", "team_b")
    .orderBy("date")
    .rowsBetween(Window.unboundedPreceding, -1)  # All previous rows for this pair
)

h2h_stats = (
    h2h_base
    .withColumn("h2h_team_a_wins",    F.sum("team_a_won").over(w_h2h))
    .withColumn("h2h_team_b_wins",    F.sum("team_b_won").over(w_h2h))
    .withColumn("h2h_draws",          F.sum("is_draw").over(w_h2h))
    .withColumn("h2h_total",          F.count("*").over(w_h2h))
    # Win rate for team_a in this h2h matchup
    .withColumn(
        "h2h_team_a_win_rate",
        F.when(
            F.col("h2h_total") > 0,
            F.col("h2h_team_a_wins") / F.col("h2h_total")
        ).otherwise(F.lit(0.33))  # Default = 1/3 (equal chance) if no history
    )
    # Fill nulls (no prior meetings)
    .fillna({"h2h_team_a_wins": 0, "h2h_team_b_wins": 0, "h2h_draws": 0, "h2h_total": 0})
    .select(
        "date", "home_team", "away_team", "team_a", "team_b",
        "h2h_team_a_wins", "h2h_team_b_wins", "h2h_draws",
        "h2h_total", "h2h_team_a_win_rate"
    )
)

print("H2H stats computed. Sample (France vs Germany matchups):")
h2h_stats.filter(
    ((F.col("home_team") == "France") & (F.col("away_team") == "Germany")) |
    ((F.col("home_team") == "Germany") & (F.col("away_team") == "France"))
).orderBy(F.col("date").desc()).show(5)

# ---- Part B: Join home team features to matches ----
# For each match, get the HOME team's feature values as of that match date
home_features = team_features_final.filter(F.col("venue") == "home")

# Rename columns: prefix with 'home_'
home_feat_renamed = home_features
for col_name in form_cols + ["matches_available"]:
    home_feat_renamed = home_feat_renamed.withColumnRenamed(col_name, f"home_{col_name}")

home_feat_renamed = home_feat_renamed.withColumnRenamed("team", "_home_team").withColumnRenamed("date", "_home_date")

# ---- Part C: Join away team features to matches ----
away_features = team_features_final.filter(F.col("venue") == "away")

away_feat_renamed = away_features
for col_name in form_cols + ["matches_available"]:
    away_feat_renamed = away_feat_renamed.withColumnRenamed(col_name, f"away_{col_name}")

away_feat_renamed = away_feat_renamed.withColumnRenamed("team", "_away_team").withColumnRenamed("date", "_away_date")

# ---- Part D: Build master feature table ----
# Join everything onto the base matches table
master = (
    df_matches.select(
        "date", "home_team", "away_team",
        "home_score", "away_score", "goal_diff",
        "result", "tournament", "neutral"
    )
    # Join home team features
    .join(
        home_feat_renamed.select(
            F.col("_home_team").alias("home_team"),
            F.col("_home_date").alias("date"),
            *[f"home_{c}" for c in form_cols + ["matches_available"]]
        ),
        on=["date", "home_team"],
        how="left"
    )
    # Join away team features
    .join(
        away_feat_renamed.select(
            F.col("_away_team").alias("away_team"),
            F.col("_away_date").alias("date"),
            *[f"away_{c}" for c in form_cols + ["matches_available"]]
        ),
        on=["date", "away_team"],
        how="left"
    )
    # Join H2H stats
    .join(
        h2h_stats.select(
            "date", "home_team", "away_team",
            "h2h_team_a_wins", "h2h_team_b_wins", "h2h_draws",
            "h2h_total", "h2h_team_a_win_rate"
        ),
        on=["date", "home_team", "away_team"],
        how="left"
    )
    # Fill remaining nulls (teams with very short history)
    .fillna(0.0)
    # Only keep matches post-1950 — earlier data lacks enough context for
    # meaningful features (FIFA WC started 1930, modern football ~1950)
    .filter(F.col("date") >= F.lit("1950-01-01").cast(T.DateType()))
    .orderBy("date")
)

match_count = master.count()
print(f"\nMaster feature table: {match_count:,} rows")
print(f"Columns: {len(master.columns)}")
master.printSchema()

## Write to PostgreSQL

We write the `match_features` table to PostgreSQL. This becomes the **single input table for the ML model** in Phase 4.

We use the **JDBC driver** bundled with PySpark to write directly to Postgres. The connection URL is read from `.env`.

> **Note:** Writing via PySpark JDBC can be slow for large datasets. For our ~40k rows it takes ~30 seconds. In production you'd increase the `numPartitions` parameter to parallelize the write.

In [ ]:
# ============================================================
# Cell 5 — Write match_features to PostgreSQL
# ============================================================
import os

# JDBC URL format for PySpark (different from SQLAlchemy format)
PG_HOST = os.getenv("POSTGRES_HOST", "localhost")
PG_PORT = os.getenv("POSTGRES_PORT", "5432")
PG_DB   = os.getenv("POSTGRES_DB",   "fifa_db")
PG_USER = os.getenv("POSTGRES_USER", "fifa_user")
PG_PASS = os.getenv("POSTGRES_PASSWORD", "fifa_pass123")

JDBC_URL = f"jdbc:postgresql://{PG_HOST}:{PG_PORT}/{PG_DB}"

jdbc_props = {
    "user":     PG_USER,
    "password": PG_PASS,
    "driver":   "org.postgresql.Driver",  # Postgres JDBC driver class
}

# ---- Download PostgreSQL JDBC driver if not already available ----
# PySpark needs the .jar file to talk to Postgres via JDBC.
# We use the popular postgresql JDBC driver.
import urllib.request
import importlib

JDBC_JAR_PATH = str(Path("..") / "drivers" / "postgresql-42.7.3.jar")
os.makedirs(str(Path("..") / "drivers"), exist_ok=True)

if not os.path.exists(JDBC_JAR_PATH):
    print("Downloading PostgreSQL JDBC driver (~1MB)...")
    urllib.request.urlretrieve(
        "https://jdbc.postgresql.org/download/postgresql-42.7.3.jar",
        JDBC_JAR_PATH
    )
    print(f"Driver saved to: {JDBC_JAR_PATH}")
else:
    print(f"JDBC driver already exists: {JDBC_JAR_PATH}")

# IMPORTANT: If SparkSession was started without the JAR, stop and restart with it.
# The JAR must be provided BEFORE SparkSession starts.
current_jars = spark.sparkContext.getConf().get("spark.jars", "")
if JDBC_JAR_PATH not in current_jars:
    print("\nRestarting SparkSession with PostgreSQL JDBC driver...")
    spark.stop()
    spark = (
        SparkSession.builder
        .appName("FIFA_WC2026_Features")
        .master("local[4]")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.driver.memory", "2g")
        .config("spark.jars", JDBC_JAR_PATH)   # <-- Include the driver JAR
        .getOrCreate()
    )
    spark.sparkContext.setLogLevel("WARN")
    print("SparkSession restarted with JDBC driver.")

    # Reload master DataFrame since session was restarted
    print("Reloading feature table from parquet...")
    df_matches = spark.read.parquet(f"{PROCESSED_DATA}/matches_clean.parquet")
    print("(Re-run Cell 3 and 4 first if you restarted; or save master to parquet below)")

# ---- Also save master features as parquet (faster than re-computing) ----
master.coalesce(1).write.mode("overwrite").parquet(
    f"{PROCESSED_DATA}/match_features.parquet"
)
print(f"Saved match_features.parquet")

# ---- Write to PostgreSQL ----
print(f"\nWriting {match_count:,} rows to PostgreSQL table 'match_features'...")
print("(This may take ~30-60 seconds via JDBC)")

(
    master
    .write
    .jdbc(
        url=JDBC_URL,
        table="match_features",
        mode="overwrite",        # Drop and recreate — safe to re-run
        properties=jdbc_props
    )
)

print("match_features table written to PostgreSQL successfully.")

## Verification — Sample Rows and Feature Statistics

Before calling Phase 2 done, we sanity-check the features:
- Win rates should be between 0 and 1
- Goal averages should be reasonable (typically 0–5)
- H2H totals should increase over time for repeated matchups
- Trend columns can be negative (declining form) or positive (improving)

In [ ]:
# ============================================================
# Cell 6 — Show sample rows and feature statistics
# ============================================================
import pandas as pd

# ---- Read back from parquet for verification ----
df_verify = spark.read.parquet(f"{PROCESSED_DATA}/match_features.parquet")

print(f"Total rows: {df_verify.count():,}")

# ---- Sample: Brazil vs Argentina matchups (most played rivalry) ----
print("\nBrazil vs Argentina feature history (last 5):")
(
    df_verify
    .filter(
        ((F.col("home_team") == "Brazil") & (F.col("away_team") == "Argentina")) |
        ((F.col("home_team") == "Argentina") & (F.col("away_team") == "Brazil"))
    )
    .select(
        "date", "home_team", "away_team", "result",
        "home_win_rate_last10", "away_win_rate_last10",
        "home_avg_goals_scored_last20", "away_avg_goals_scored_last20",
        "h2h_total", "h2h_team_a_win_rate"
    )
    .orderBy(F.col("date").desc())
    .show(5, truncate=False)
)

# ---- Summary statistics for key feature columns ----
print("\nFeature statistics (describe):")
feature_stat_cols = [
    "home_win_rate_last10", "away_win_rate_last10",
    "home_avg_goals_scored_last20", "away_avg_goals_scored_last20",
    "home_avg_goals_conceded_last20", "away_avg_goals_conceded_last20",
    "home_goal_diff_trend", "away_goal_diff_trend",
    "home_ranking_proxy", "away_ranking_proxy",
    "h2h_total",
]

df_verify.select(feature_stat_cols).describe().show(truncate=False)

# ---- Class distribution (target variable check) ----
print("Result distribution (target variable):")
df_verify.groupBy("result").count().orderBy("result").show()

# ---- Feature completeness check ----
# Count nulls — there should be very few after our fillna(0.0)
print("\nNull check on key feature columns:")
null_counts = {
    c: df_verify.filter(F.col(c).isNull()).count()
    for c in feature_stat_cols
}
for col_name, count in null_counts.items():
    status = "OK" if count == 0 else f"WARNING: {count} nulls"
    print(f"  {col_name:<40} {status}")

spark.stop()
print("\nPhase 2 complete. SparkSession stopped.")
print("match_features table is ready for dbt (Phase 3) and ML (Phase 4).")